# 02 — The Evaluator Defect Trichotomy
## Why Two-Rate Dashboards Lie and How Defect Partitions Uncover Truth

**Series:** `finite_obstruction_calculus` / `012_lggr_i_evaluated_path_kernels`  
**Author:** Jeremy H. Carroll  
**Status:** Living Educational Supplement  
**Anchor Tag:** `sealed-derivation-v1` (`3100c4e`, cut 2026-08-17)

---

> ### **Claim-Typing Legend**
> - **[Pinned]**: Verified directly by executable Python functions in `code/epk_core.py` and passing test suites.
> - **[Proved]**: Mechanized in Lean 4 under `PaperAE004` (reported kernel axioms: `propext`, `Classical.choice`, `Quot.sound` or axiom-free).
> - **[Declared]**: An operational definition, architectural parameter, or illustrative teaching model chosen by the author.
> - **[Open]**: A mathematical or empirical question not discharged in this package.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. The Three Semantic States of Verification

In any multi-stage system with binary local checks ($E_1, E_2 \in \{0, 1\}$) and a binary global check ($E_g \in \{0, 1\}$), the Evaluator Gluing Defect can take **only three possible values**:
$$\Delta_E = E_g - E_1 E_2 \in \{-1, 0, +1\}$$

This mathematical partition is called the **Defect Trichotomy** (§1.5, §5):

| Defect ($\Delta_E$) | Name | Condition | Practical Real-World Meaning |
| :---: | :--- | :--- | :--- |
| **$-1$** | **Negative Gluing Defect (NGD)** | $E_1 E_2 = 1$ and $E_g = 0$ | **Local Green, Global Red.** Sub-agents passed every internal test, but their composed output failed integration. |
| **$0$** | **Zero Defect Residual (ZDR)** | $E_g = E_1 E_2$ | **Aligned Evaluation.** Local tests accurately predicted integration (both passed $1=1$, or at least one failed $0=0$). |
| **$+1$** | **Positive Gluing Defect (PGD)** | $E_1 E_2 = 0$ and $E_g = 1$ | **Dynamic Rescue / Error Tolerance.** A local sub-check failed, but runtime fallbacks, self-healing, or downstream agents fixed the problem! |

Every single execution trace falls cleanly into exactly one of these three buckets.


## 2. The Two-Rate Dashboard Fallacy (§1.5)

Most AI engineering teams look at an executive dashboard that tracks two numbers:
1. **Local Pass Rate (LPR):** How often do sub-agents pass their step checks?
2. **Global Pass Rate (GPR):** How often does the overall task succeed?

Because $\mathrm{GPR} - \mathrm{LPR} = \bar{\Delta}_E$, many teams assume that tracking these two rates is enough to diagnose their pipeline.

**This is mathematically false.**

### The Proof: Two Systems with Identical Dashboards
Consider two different 10-trace runs:

- **System A (Pure Integration Blindness):**
  - Traces: 6 local passes, 2 global passes.
  - Defect breakdown: $4$ NGD traces ($\Delta_E = -1$), $0$ PGD traces ($\Delta_E = +1$), $6$ ZDR traces ($\Delta_E = 0$).
  - $\mathrm{LPR} = 0.60, \quad \mathrm{GPR} = 0.20, \quad \bar{\Delta}_E = -0.40$.

- **System B (High-Volatility with Dynamic Rescues):**
  - Traces: 6 local passes, 2 global passes.
  - Defect breakdown: $5$ NGD traces ($\Delta_E = -1$), $1$ PGD trace ($\Delta_E = +1$), $4$ ZDR traces ($\Delta_E = 0$).
  - Mean defect: $\bar{\Delta}_E = \mathrm{PGD} - \mathrm{NGD} = 0.10 - 0.50 = -0.40$.
  - $\mathrm{LPR} = 0.60, \quad \mathrm{GPR} = 0.20, \quad \bar{\Delta}_E = -0.40$.

On the dashboard, **System A and System B look 100% identical!**
Both show $\mathrm{LPR} = 60\%$ and $\mathrm{GPR} = 20\%$.
Yet System B has a working dynamic recovery mechanism ($\mathrm{PGD} = 10\%$) while suffering from even worse baseline integration failure ($\mathrm{NGD} = 50\%$). A two-rate dashboard cannot see this difference!


In [ ]:
# Construct System A traces (4 NGD, 0 PGD, 6 ZDR)
traces_a = [
    # 2 Aligned passes (ZDR)
    EvaluatorVerdictTriple(1, 1, 1),
    EvaluatorVerdictTriple(1, 1, 1),
    # 4 Integration failures (NGD: E1*E2=1, Eg=0)
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    # 4 Aligned failures (ZDR: E1*E2=0, Eg=0)
    EvaluatorVerdictTriple(1, 0, 0),
    EvaluatorVerdictTriple(0, 1, 0),
    EvaluatorVerdictTriple(0, 0, 0),
    EvaluatorVerdictTriple(0, 0, 0),
]

# Construct System B traces (5 NGD, 1 PGD, 4 ZDR)
traces_b = [
    # 1 Aligned pass (ZDR)
    EvaluatorVerdictTriple(1, 1, 1),
    # 5 Integration failures (NGD: E1*E2=1, Eg=0)
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    EvaluatorVerdictTriple(1, 1, 0),
    # 1 Dynamic rescue (PGD: E1*E2=0, Eg=1)
    EvaluatorVerdictTriple(1, 0, 1),
    # 3 Aligned failures (ZDR: E1*E2=0, Eg=0)
    EvaluatorVerdictTriple(0, 1, 0),
    EvaluatorVerdictTriple(0, 0, 0),
    EvaluatorVerdictTriple(0, 0, 0),
]

metrics_a = EPKMetrics.compute(traces_a)
metrics_b = EPKMetrics.compute(traces_b)

comparison = pd.DataFrame([
    {"System": "System A (Pure Blindness)", "LPR": f"{metrics_a.lpr:.1%}", "GPR": f"{metrics_a.gpr:.1%}", "bar(Delta)": f"{metrics_a.mean_delta_e:.2f}", "NGD Rate": f"{metrics_a.ngd_all:.1%}", "PGD Rate": f"{metrics_a.pgd:.1%}", "ZDR Rate": f"{metrics_a.zdr:.1%}"},
    {"System": "System B (Dynamic Rescues)", "LPR": f"{metrics_b.lpr:.1%}", "GPR": f"{metrics_b.gpr:.1%}", "bar(Delta)": f"{metrics_b.mean_delta_e:.2f}", "NGD Rate": f"{metrics_b.ngd_all:.1%}", "PGD Rate": f"{metrics_b.pgd:.1%}", "ZDR Rate": f"{metrics_b.zdr:.1%}"}
])

print("=== TWO SYSTEMS, IDENTICAL DASHBOARD, DIFFERENT REALITIES ===")
comparison
# Notebook-level assertions for the headline contrast:
assert metrics_a.lpr == metrics_b.lpr == 0.60
assert metrics_a.gpr == metrics_b.gpr == 0.20
assert metrics_a.mean_delta_e == metrics_b.mean_delta_e == -0.40
assert metrics_a.ngd_all == 0.40 and metrics_b.ngd_all == 0.50
assert metrics_a.pgd == 0.00 and metrics_b.pgd == 0.10
assert metrics_a.zdr == 0.60 and metrics_b.zdr == 0.40
assert metrics_a.egd == 0.40 and metrics_b.egd == 0.60
assert metrics_a.egd != metrics_b.egd



In [ ]:
# Visualizing the Diagnostic Plane and Defect Shifts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

# Panel 1: The (LPR, GPR) Diagnostic Plane
lpr_vals = np.linspace(0, 1, 100)
gpr_vals = np.linspace(0, 1, 100)
L, G = np.meshgrid(lpr_vals, gpr_vals)
Delta_grid = G - L

cs = ax1.contourf(L, G, Delta_grid, levels=12, cmap='coolwarm', alpha=0.75)
cbar = fig.colorbar(cs, ax=ax1)
cbar.set_label(r"Mean Defect $\bar{\Delta}_E = \mathrm{GPR} - \mathrm{LPR}$", fontsize=10, fontweight='bold')

# Diagonal line (Zero defect balance)
ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label=r"Aligned Balance ($\bar{\Delta}_E = 0$)")

# Mark System A and B
ax1.scatter([0.6], [0.2], color='black', s=120, zorder=5, label="Both Systems (LPR=0.6, GPR=0.2)")
ax1.annotate("Identical Dashboard\nPoint (0.60, 0.20)", (0.6, 0.2), textcoords="offset points", xytext=(-80, -25),
             bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.7),
             arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0"))

ax1.fill_between([0, 1], [0, 0], [0, 1], color='red', alpha=0.15, label="LGGR Zone (LPR > GPR)")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.set_xlabel("Local Pass Rate (LPR)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Global Pass Rate (GPR)", fontsize=11, fontweight='bold')
ax1.set_title("(a) The LGGR Diagnostic Plane", fontsize=11, fontweight='bold')
ax1.legend(loc="upper left", fontsize=8.5)
ax1.grid(True, linestyle=':', alpha=0.6)

# Panel 2: The Real Defect Trichotomy Comparison
categories = ['NGD (Defect)', 'PGD (Rescue)', 'ZDR (Aligned)']
width = 0.35
x = np.arange(len(categories))

rects1 = ax2.bar(x - width/2, [metrics_a.ngd_all, metrics_a.pgd, metrics_a.zdr], width,
                label='System A', color='#e41a1c', alpha=0.85, edgecolor='black')
rects2 = ax2.bar(x + width/2, [metrics_b.ngd_all, metrics_b.pgd, metrics_b.zdr], width,
                label='System B', color='#377eb8', alpha=0.85, edgecolor='black')

ax2.set_ylabel('Rate Fraction', fontsize=11, fontweight='bold')
ax2.set_title('(b) Unmasking the Defect Distribution', fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax2.set_ylim(0, 0.75)
ax2.legend(fontsize=10)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

# Add labels
for bar in rects1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.0%}", ha='center', fontsize=9, fontweight='bold')
for bar in rects2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.0%}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Machine-Checked Proof in Lean 4

This two-rate dashboard insufficiency and quotient separation is formalized in Lean 4 under `PaperAE004`:

- **Theorem `same_lpr_gpr_and_delta_sum_different_egd` (`13_two_rate_dashboard_insufficiency.lean`):**  
  Proves constructively that the two-rate dashboard map $(E_1, E_2, E_g) \mapsto (\mathrm{LPR}, \mathrm{GPR})$ is strictly many-to-one and insufficient to determine the Evaluator Gluing Defect or the NGD/PGD/ZDR partition: there exist distinct trace distributions with identical LPR and GPR but different defect measures (Claim E9 / A6g).
- **Theorem `local_dashboard_zero_identifies_all_green_and_lggr` (`08_primitive_difference_bridge.lean`):**  
  Proves non-factorization through the local-dashboard primitive-difference quotient: the local dashboard does not carry the global evaluator $E_g$ or the signed evaluator defect $\Delta_E$ (Claim E4 / A6b).

No custom axioms are invoked (`propext`, `Classical.choice`, `Quot.sound` only).


## 4. Mandatory Claim Hygiene: CAN-Show vs CANNOT-Show

### What this model CAN show:
- Shows that tracking only local and global pass rates $(\mathrm{LPR}, \mathrm{GPR})$ discards vital information about the internal defect structure.
- Shows that the defect trichotomy $(\mathrm{NGD}, \mathrm{PGD}, \mathrm{ZDR})$ is necessary to distinguish silent integration failure from active error recovery.
- Shows that two systems with identical mean defect $\bar{\Delta}_E$ can have completely different operational reliability profiles.

### What this model CANNOT show:
- Does **not** show that dynamic rescue (PGD) is always desirable (a system with high PGD might simply have broken, noisy local checks that fail constantly for no reason).
- Does **not** claim that real-world multi-agent pipelines always have stationary defect rates across different tasks.


## 5. Student Exercises

### Exercise 1: Defect Rate Constraints
If an engineering team measures $\mathrm{NGD} = 0.30$ and $\mathrm{PGD} = 0.10$ on a benchmark:
1. What is the fraction of aligned traces ($\mathrm{ZDR}$)?
2. What is the mean defect $\bar{\Delta}_E$?
3. If their local pass rate is $\mathrm{LPR} = 75\%$, what is their true global pass rate $\mathrm{GPR}$?

<details>
<summary><b>Click to see the solution for Exercise 1</b></summary>

1. From Trichotomy Normal Form:  
   $$\mathrm{ZDR} = 1 - (\mathrm{NGD} + \mathrm{PGD}) = 1 - (0.30 + 0.10) = 0.60 \quad (60\%)$$
2. From Signed Defect Balance:  
   $$\bar{\Delta}_E = \mathrm{PGD} - \mathrm{NGD} = 0.10 - 0.30 = -0.20$$
3. From the Pass-Rate Gap Identity:  
   $$\mathrm{GPR} = \mathrm{LPR} + \bar{\Delta}_E = 0.75 + (-0.20) = 0.55 \quad (55\%)$$
</details>

### Exercise 2: The Fully Aligned Ideal
What mathematical condition on the trichotomy must hold for the local dashboard to perfectly equal the global dashboard ($\mathrm{LPR} = \mathrm{GPR}$)? Does this require zero failures?

<details>
<summary><b>Click to see the solution for Exercise 2</b></summary>

- $\mathrm{GPR} = \mathrm{LPR} \iff \bar{\Delta}_E = 0 \iff \mathrm{PGD} = \mathrm{NGD}$.
- In the ideal alignment case, $\mathrm{PGD} = 0$ and $\mathrm{NGD} = 0$, so $\mathrm{ZDR} = 100\%$.
- **Does it require zero failures? NO!** If every trace either passes both local and global ($1,1,1$) OR fails both local and global ($0,0,0$ or $1,0,0$), then $\Delta_E = 0$ everywhere. You can have a $0\%$ pass rate that is perfectly aligned ($\mathrm{ZDR}=100\%$).
</details>


---
## Anchors & Provenance

- **Paper Sections:** §1.5 (Two-Rate Dashboard Insufficiency), §5 (Defect Metrics and Identifications), §8.3 (Sample Identifications).
- **Claim Ledger IDs:** D1 (Delta trichotomy), E1 (NGD/PGD/ZDR partition), E5 (Semantic normal forms), E7 (EGD decomposition).
- **Code Functions:** `epk_core.EPKMetrics`, `epk_core.calculate_evaluator_defect`.
- **Lean 4 Mechanization:**
  - `PaperAE004.deltaEval_range` in `lean4/01_local_green_counterexample.lean`
  - `PaperAE004.E1_ngd_pgd_zdr_partition` in `lean4/lggr_i_evaluated_path_kernels.lean`
  - `PaperAE004.E8_signed_defect_balance` in `lean4/lggr_i_evaluated_path_kernels.lean`
- **Next Step:** Open [`03_boundary_slices_and_composition.ipynb`](03_boundary_slices_and_composition.ipynb) to see how agent trajectories move through state space and how kernels compose across intermediate checkpoints.
